# Transformer Components Test
Notebook này dùng để kiểm tra thực tế dữ liệu đầu vào (Input) và đầu ra (Output) của từng thành phần trong kiến trúc Transformer.

---

In [3]:
import torch
import os
import sys

# Để import được các module từ src/
sys.path.append(os.path.abspath('..'))

## 0. Data & Tokenizer
Kiểm tra cách bộ Tokenizer (SentencePiece) chuyển đổi văn bản thành mã số (IDs).

In [4]:
import sentencepiece as spm
from src.utils import load_config

cfg = load_config('../configs/config.yaml')
tokenizer_path = os.path.join('..', cfg['tokenizer']['model_path'])

if os.path.exists(tokenizer_path):
    sp = spm.SentencePieceProcessor()
    sp.load(tokenizer_path)
    
    text = "Transformer là kiến trúc mạng nơ-ron mạnh mẽ cho dịch máy."
    ids = sp.encode_as_ids(text)
    pieces = sp.encode_as_pieces(text)
    
    print(f"Văn bản gốc: {text}")
    print(f"Các mảnh (pieces): {pieces}")
    print(f"Các mã số (IDs): {ids}")
    print(f"Giải mã ngược lại: {sp.decode_ids(ids)}")

Văn bản gốc: Transformer là kiến trúc mạng nơ-ron mạnh mẽ cho dịch máy.
Các mảnh (pieces): ['▁Trans', 'form', 'er', '▁là', '▁kiến', '▁trúc', '▁mạng', '▁nơ', '-', 'ron', '▁mạnh', '▁mẽ', '▁cho', '▁dịch', '▁máy', '.']
Các mã số (IDs): [6928, 1663, 17, 56, 1702, 3011, 1759, 13048, 31871, 1287, 1274, 3988, 145, 1323, 910, 31847]
Giải mã ngược lại: Transformer là kiến trúc mạng nơ-ron mạnh mẽ cho dịch máy.


## 0.1 Data Pipeline (DataLoader)
Kiểm tra cách dữ liệu được tải lên, làm sạch và đóng gói thành Batch.

In [5]:
from src.data.dataset import get_dataloader

# Thử nghiệm với tập test hoặc val để nhanh hơn
data_path = os.path.join('..', cfg['dataset']['test_csv'])
spm_path = os.path.join('..', cfg['tokenizer']['model_path'])

if os.path.exists(data_path) and os.path.exists(spm_path):
    dataset, loader = get_dataloader(
        data_sources=data_path,
        spm_model_path=spm_path,
        batch_size=4,
        pad_id=0,
        shuffle=True,
        num_workers=0 # Laptop/Windows thường lỗi đa luồng trong notebook
    )
    
    # Lấy thử 1 batch
    batch = next(iter(loader))
    
    print(f"Batch Source shape: {batch['src'].shape}")
    print(f"Batch Target shape: {batch['tgt'].shape}")
    print(f"Ví dụ 1 dòng trong Batch (IDs):\n{batch['src'][0]}")
else:
    print("⚠️ Thiếu file CSV dữ liệu hoặc Tokenizer model để chạy Pipeline test.")

KeyError: 'dataset'

## 1. Utilities: Masking
Kiểm tra các hàm tạo mặt nạ (Mask).

In [ ]:
from src.utils.mask import create_src_mask, create_tgt_mask

# Ví dụ: Batch=1, SeqLen=5. Số 0 là padding
src = torch.tensor([[1, 2, 3, 0, 0]])
src_mask = create_src_mask(src, pad_idx=0)
print(f"Source Tensor: {src}")
print(f"Source Mask shape: {src_mask.shape}")
print(f"Source Mask:\n{src_mask}")

print("-" * 30)

tgt = torch.tensor([[5, 6, 0]])
tgt_mask = create_tgt_mask(tgt, pad_idx=0)
print(f"Target Tensor: {tgt}")
print(f"Target Mask shape: {tgt_mask.shape}")
print(f"Target Mask (Layer 0):\n{tgt_mask[0, 0]}")

## 2. Layers: Attention, FeedForward, Positional Encoding
Kiểm tra các thành phần cơ bản xử lý vector.

In [ ]:
from src.layer.attention import MultiHeadAttention
from src.layer.feed_forward import FeedForward
from src.layer.positional_encoding import PositionalEncoding

batch_size, seq_len, d_model = 2, 10, 512
x = torch.randn(batch_size, seq_len, d_model)

# 2.1 Multi-Head Attention
mha = MultiHeadAttention(d_model=512, num_heads=8)
out_mha = mha(x, x, x) # Self-attention
print(f"MHA Input: {x.shape} -> Output: {out_mha.shape}")

# 2.2 FeedForward
ff = FeedForward(d_model=512, d_ff=2048)
out_ff = ff(x)
print(f"FF Input: {x.shape} -> Output: {out_ff.shape}")

# 2.3 Positional Encoding
pe = PositionalEncoding(d_model=512, max_len=100)
out_pe = pe(x)
print(f"PE Input: {x.shape} -> Output: {out_pe.shape}")

## 3. Core: Encoder & Decoder
Kiểm tra các khối xử lý lớp chồng lớp.

In [ ]:
from src.layer.encoder import Encoder
from src.layer.decoder import Decoder

src_ids = torch.randint(0, 1000, (2, 10))
tgt_ids = torch.randint(0, 1000, (2, 8))

# 3.1 Encoder
encoder = Encoder(vocab_size=1000, d_model=512, num_layers=2, num_heads=8)
enc_out = encoder(src_ids)
print(f"Encoder Input: {src_ids.shape} (IDs) -> Output: {enc_out.shape} (Context)")

# 3.2 Decoder
decoder = Decoder(vocab_size=1000, d_model=512, num_layers=2, num_heads=8)
dec_out = decoder(tgt_ids, enc_out)
print(f"Decoder Input: {tgt_ids.shape} (IDs) + {enc_out.shape} (Context) -> Output: {dec_out.shape} (Logits)")

## 4. Full Model: Transformer
Kiểm tra luồng xử lý từ đầu đến cuối.

In [ ]:
from src.model import Transformer

model = Transformer(
    src_vocab=1000, 
    tgt_vocab=1000, 
    d_model=512, 
    num_layers=2, 
    num_heads=8
)

logits = model(src_ids, tgt_ids)
print(f"Transformer Full Forward Path:")
print(f"- Input Src: {src_ids.shape}")
print(f"- Input Tgt: {tgt_ids.shape}")
print(f"- Output Logits: {logits.shape}")